# 🛡️ DocuGuard+ — Google Colab GPU Backend

> **Run your LLM humanization on Colab's free T4 GPU (16 GB VRAM) and connect it to your local DocuGuard+ app via Cloudflare tunnel.**

## What This Notebook Does

| Step | Action |
|------|--------|
| 1 | Installs **Ollama**, **Cloudflare Tunnel**, and **MCP Debug Server** deps |
| 2 | Starts the **Ollama server** (GPU-accelerated) |
| 3 | Pulls your chosen LLM model (LLama3 / Mistral) |
| 4 | Verifies **GPU usage** and runs inference test |
| 5 | Starts **Cloudflare tunnel** → gives you a public Ollama URL |
| 6 | Starts **MCP Debug Server** → lets Copilot monitor & debug remotely |
| 7 | **Keep-alive** loop monitors both services |

## Architecture
```
Google Colab (T4 GPU)
    ├── Ollama Server (:11434)
    │     └── Cloudflare Tunnel → Ollama URL (for DocuGuard+ app)
    │
    ├── MCP Debug Server (:8000)
    │     ├── POST /run_cell        — Execute Python code remotely
    │     ├── GET  /get_last_error  — Retrieve last error traceback
    │     ├── GET  /get_gpu_usage   — nvidia-smi output
    │     ├── GET  /get_ollama_status — Ollama health + models
    │     ├── GET  /get_logs        — Read Ollama server logs
    │     └── Cloudflare Tunnel → MCP URL (for VS Code Copilot)
    │
VS Code (Local)
    └── colab_bridge.py → Calls MCP endpoints via tunnel
          └── Copilot can run this to monitor/debug Colab
```

## Prerequisites
- Google Colab account (free tier works)
- **Runtime → Change runtime type → T4 GPU**
- Your local machine running DocuGuard+ Streamlit app

## Performance
| Setup | Speed |
|-------|-------|
| Local CPU | ~5-15 tokens/sec |
| **Colab T4 GPU** | **~40-80 tokens/sec (5-8x faster)** |

## Step 1: Install Ollama & Cloudflare Tunnel
This installs the Ollama LLM runtime and the Cloudflare tunnel binary on the Colab VM.

In [ ]:
%%bash
# Install zstd (required by Ollama installer)
apt-get update -qq && apt-get install -y -qq zstd > /dev/null 2>&1
echo "✅ zstd installed"

# Install Ollama
curl -fsSL https://ollama.com/install.sh | sh

# Install Cloudflare tunnel
wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
  -O /usr/local/bin/cloudflared
chmod +x /usr/local/bin/cloudflared

# Install MCP Debug Server dependencies
pip install -q fastapi uvicorn nest_asyncio psutil

echo "✅ Ollama, Cloudflare, and MCP Debug Server installed successfully"

## Step 2: Start Ollama Server (GPU-Accelerated)
Launches the Ollama server bound to `0.0.0.0:11434` so the tunnel can reach it.

In [ ]:
import subprocess, os, time

# Start Ollama server with GPU support (use full path)
OLLAMA_BIN = "/usr/local/bin/ollama"

env = {**os.environ, "OLLAMA_HOST": "0.0.0.0:11434"}
ollama_proc = subprocess.Popen(
    [OLLAMA_BIN, "serve"],
    env=env,
    stdout=open("/tmp/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)
time.sleep(5)

# Verify it's running
import requests
try:
    r = requests.get("http://localhost:11434/api/tags", timeout=5)
    print(f"✅ Ollama server is running (status {r.status_code})")
except Exception as e:
    print(f"⏳ Server not ready yet: {e}")
    print("   Waiting 10 more seconds...")
    time.sleep(10)
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=5)
        print(f"✅ Ollama server is running (status {r.status_code})")
    except Exception as e2:
        print(f"❌ Server failed to start: {e2}")
        print("   Check logs: !cat /tmp/ollama.log")

## Step 3: Pull LLM Model

Choose your model:
- **`llama3`** — 4.7 GB, best quality output (recommended)
- **`mistral`** — 4.1 GB, faster, good quality

Run **one** of the cells below.

In [ ]:
# ===== Option A: LLama 3 (Recommended — best output quality) =====
!/usr/local/bin/ollama pull llama3
print("✅ LLama 3 model ready")

In [ ]:
# ===== Option B: Mistral (Faster, slightly smaller) =====
# Uncomment and run this cell instead if you prefer Mistral
# !/usr/local/bin/ollama pull mistral
# print("✅ Mistral model ready")

## Step 4: Verify GPU Usage
Confirm the T4 GPU is detected and Ollama is using it.

In [ ]:
!echo "=== GPU Info ==="
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader

!echo ""
!echo "=== Loaded Models ==="
!/usr/local/bin/ollama list

!echo ""
!echo "=== Quick Inference Test ==="
import requests, time
start = time.time()
r = requests.post("http://localhost:11434/api/generate", json={
    "model": "llama3",
    "prompt": "Say hello in one sentence.",
    "stream": False,
})
elapsed = time.time() - start
resp = r.json().get("response", "")
print(f"Response: {resp[:200]}")
print(f"⏱️ Inference time: {elapsed:.2f}s")

## Step 5: Start Cloudflare Tunnel 🚀

This creates a **free public URL** that forwards to your Colab's Ollama server.

**Copy the URL** that appears below and paste it into DocuGuard+'s sidebar under **🔗 LLM Backend**.

In [ ]:
import subprocess, re, time, threading
from IPython.display import display, Markdown, HTML

# Start Cloudflare tunnel in background
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:11434", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

# Wait for the tunnel URL to appear
tunnel_url = None
deadline = time.time() + 30

while time.time() < deadline:
    line = tunnel_proc.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    match = re.search(r"(https://[a-z0-9-]+\.trycloudflare\.com)", line)
    if match:
        tunnel_url = match.group(1)
        break

if tunnel_url:
    display(HTML(f"""
    <div style="background: #1a1a2e; border: 3px solid #00d4ff; border-radius: 12px;
                padding: 20px; margin: 10px 0; text-align: center;">
        <h2 style="color: #00d4ff; margin: 0 0 10px 0;">🚀 Tunnel Active!</h2>
        <p style="color: #e0e0e0; font-size: 14px; margin: 5px 0;">
            Copy this URL and paste it into DocuGuard+ sidebar → <b>🔗 LLM Backend</b>
        </p>
        <code style="background: #0d1117; color: #58a6ff; padding: 12px 24px;
                    border-radius: 8px; font-size: 18px; display: inline-block;
                    margin: 10px 0; user-select: all; cursor: pointer;">
            {tunnel_url}
        </code>
        <p style="color: #8b949e; font-size: 12px; margin: 10px 0 0 0;">
            ⚠️ This URL is temporary and changes each time you restart.
            Keep this notebook running!
        </p>
    </div>
    """))
else:
    print("❌ Tunnel failed to start within 30 seconds.")
    print("Check logs: !cat /tmp/ollama.log")

## Step 6: Start MCP Debug Server 🔧

This runs a lightweight **FastAPI server** on port 8000 that exposes debugging endpoints.
Your VS Code Copilot agent can call these remotely to:
- **Execute code** on Colab
- **Read error tracebacks** without copy-pasting
- **Check GPU memory** usage
- **Read Ollama logs** for troubleshooting
- **Check Ollama health** and loaded models

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import traceback, psutil, subprocess, os, threading, time

# ─── MCP Debug Server ───────────────────────────────────────
mcp_app = FastAPI(title="DocuGuard+ Colab MCP Debug Server")
mcp_app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

# Shared state
_last_error = None
_execution_history = []

class RunCode(BaseModel):
    code: str

class CellResult(BaseModel):
    status: str
    output: str = ""
    error: str = ""

# ─── Endpoints ────────────────────────────────────────────

@mcp_app.get("/health")
def health():
    return {"status": "ok", "service": "docuguard-colab-mcp", "uptime_min": int((time.time() - _mcp_start_time) / 60)}

@mcp_app.post("/run_cell")
def run_cell(payload: RunCode):
    """Execute arbitrary Python code on the Colab runtime."""
    global _last_error
    import io, contextlib
    buf = io.StringIO()
    try:
        with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
            exec(payload.code, globals())
        output = buf.getvalue()
        _last_error = None
        _execution_history.append({"code": payload.code[:200], "status": "success"})
        return {"status": "success", "output": output[:5000]}
    except Exception as e:
        tb = traceback.format_exc()
        _last_error = tb
        _execution_history.append({"code": payload.code[:200], "status": "error", "error": str(e)})
        return {"status": "error", "error": str(e), "traceback": tb}

@mcp_app.get("/get_last_error")
def get_last_error():
    """Return the last error traceback from run_cell, if any."""
    return {"last_error": _last_error, "has_error": _last_error is not None}

@mcp_app.get("/get_gpu_usage")
def get_gpu_usage():
    """Return nvidia-smi output with GPU memory stats."""
    try:
        result = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.total,memory.used,memory.free,utilization.gpu",
             "--format=csv,noheader,nounits"],
            timeout=10
        ).decode().strip()
        parts = [x.strip() for x in result.split(",")]
        return {
            "gpu_name": parts[0] if len(parts) > 0 else "unknown",
            "memory_total_mb": parts[1] if len(parts) > 1 else "?",
            "memory_used_mb": parts[2] if len(parts) > 2 else "?",
            "memory_free_mb": parts[3] if len(parts) > 3 else "?",
            "gpu_utilization_pct": parts[4] if len(parts) > 4 else "?",
            "raw": result,
        }
    except Exception as e:
        return {"error": str(e), "gpu_name": "unavailable"}

@mcp_app.get("/get_ollama_status")
def get_ollama_status():
    """Check if Ollama is running and list loaded models."""
    import requests as _req
    try:
        r = _req.get("http://localhost:11434/api/tags", timeout=5)
        models = r.json().get("models", [])
        model_names = [m.get("name", "unknown") for m in models]
        return {"running": True, "models": model_names, "model_count": len(models)}
    except Exception as e:
        return {"running": False, "error": str(e), "models": []}

@mcp_app.get("/get_logs")
def get_logs(lines: int = 50):
    """Return the last N lines of the Ollama server log."""
    try:
        with open("/tmp/ollama.log", "r") as f:
            all_lines = f.readlines()
        tail = all_lines[-lines:]
        return {"lines": len(tail), "log": "".join(tail)}
    except FileNotFoundError:
        return {"error": "Log file not found at /tmp/ollama.log", "log": ""}
    except Exception as e:
        return {"error": str(e), "log": ""}

@mcp_app.get("/get_system_info")
def get_system_info():
    """Return CPU, RAM, and disk usage."""
    import shutil
    disk = shutil.disk_usage("/")
    return {
        "cpu_percent": psutil.cpu_percent(interval=1),
        "ram_total_gb": round(psutil.virtual_memory().total / 1e9, 1),
        "ram_used_gb": round(psutil.virtual_memory().used / 1e9, 1),
        "ram_percent": psutil.virtual_memory().percent,
        "disk_total_gb": round(disk.total / 1e9, 1),
        "disk_free_gb": round(disk.free / 1e9, 1),
    }

@mcp_app.get("/get_execution_history")
def get_execution_history(last: int = 10):
    """Return the last N execution results."""
    return {"history": _execution_history[-last:], "total": len(_execution_history)}

# ─── Start server in background thread ───────────────────

_mcp_start_time = time.time()

def _run_mcp():
    import uvicorn
    uvicorn.run(mcp_app, host="0.0.0.0", port=8000, log_level="warning")

mcp_thread = threading.Thread(target=_run_mcp, daemon=True)
mcp_thread.start()
time.sleep(2)

# Verify
import requests as _req
try:
    r = _req.get("http://localhost:8000/health", timeout=5)
    print(f"✅ MCP Debug Server running on port 8000 — {r.json()}")
except Exception as e:
    print(f"❌ MCP server failed to start: {e}")

## Step 7: Expose MCP Debug Server via Tunnel 🌐

This creates a **second Cloudflare tunnel** (port 8000) for the MCP Debug Server.

**Copy the MCP URL** and use it with `colab_bridge.py` on your local machine:
```bash
python colab_bridge.py --url https://YOUR-MCP-URL.trycloudflare.com status
```

In [ ]:
import subprocess, re, time
from IPython.display import display, HTML

# Start second Cloudflare tunnel for MCP server (port 8000)
mcp_tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

# Wait for the tunnel URL
mcp_tunnel_url = None
deadline = time.time() + 30

while time.time() < deadline:
    line = mcp_tunnel_proc.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    match = re.search(r"(https://[a-z0-9-]+\.trycloudflare\.com)", line)
    if match:
        mcp_tunnel_url = match.group(1)
        break

if mcp_tunnel_url:
    display(HTML(f"""
    <div style="background: #0d1117; border: 3px solid #f0883e; border-radius: 12px;
                padding: 20px; margin: 10px 0; text-align: center;">
        <h2 style="color: #f0883e; margin: 0 0 10px 0;">🔧 MCP Debug Server Active!</h2>
        <p style="color: #e0e0e0; font-size: 14px; margin: 5px 0;">
            Use this URL with <code>colab_bridge.py</code> from your local machine:
        </p>
        <code style="background: #161b22; color: #f0883e; padding: 12px 24px;
                    border-radius: 8px; font-size: 16px; display: inline-block;
                    margin: 10px 0; user-select: all; cursor: pointer;">
            {mcp_tunnel_url}
        </code>
        <p style="color: #8b949e; font-size: 13px; margin: 10px 0 0 0;">
            <b>Local usage:</b> <code style="color:#79c0ff">python colab_bridge.py --url {mcp_tunnel_url} status</code>
        </p>
        <hr style="border-color: #30363d; margin: 12px 0;">
        <table style="margin: auto; color: #c9d1d9; font-size: 13px; border-collapse: collapse;">
            <tr><td style="padding: 4px 12px; text-align: left;">🚀 Ollama Tunnel:</td>
                <td style="padding: 4px 12px; color: #58a6ff;">{tunnel_url}</td></tr>
            <tr><td style="padding: 4px 12px; text-align: left;">🔧 MCP Debug Tunnel:</td>
                <td style="padding: 4px 12px; color: #f0883e;">{mcp_tunnel_url}</td></tr>
        </table>
    </div>
    """))
else:
    print("❌ MCP tunnel failed to start within 30 seconds.")
    print("   Try manually: !cloudflared tunnel --url http://localhost:8000")

## Step 8: Keep Alive ♾️

**Run this cell and leave it running.** It keeps the Colab session, Ollama, and MCP server alive.

The session will auto-disconnect after ~90 minutes of inactivity on free tier.
To extend, interact with the notebook periodically.

In [ ]:
import time, requests
from IPython.display import clear_output

print("♾️ Keep-alive running. Do NOT stop this cell.")
print(f"🚀 Ollama  Tunnel: {tunnel_url}")
print(f"🔧 MCP Debug Tunnel: {mcp_tunnel_url}")
print("-" * 60)

elapsed_min = 0
while True:
    time.sleep(60)
    elapsed_min += 1

    # Ollama heartbeat
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=5)
        ollama_status = f"✅ Ollama OK ({len(r.json().get('models', []))} models)"
    except Exception:
        ollama_status = "⚠️ Ollama not responding"

    # MCP heartbeat
    try:
        r = requests.get("http://localhost:8000/health", timeout=5)
        mcp_status = f"✅ MCP OK (uptime {r.json().get('uptime_min', '?')}min)"
    except Exception:
        mcp_status = "⚠️ MCP not responding"

    # Print periodic status (every 5 minutes)
    if elapsed_min % 5 == 0:
        print(f"[{elapsed_min}min] {ollama_status} | {mcp_status}")